# Feature Engineering

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("data/raw/india_startup_master_dataset.csv")

df.head()

,year,state,industry,startup_count,nsva,india_internet_penetration,india_population,state_total_startups,sector_share,sector_share_pct,startup_growth_pct,india_internet_growth_pp,india_total_startups,industry_share_national
0,2017,Andhra Pradesh,AI,2.0,645027.0,18.200001,1.359657e+09,103.0,0.019417,1.941748,2475.0,1.700001,5473.0,0.014069
1,2017,Andhra Pradesh,AR VR (Augmented + Virtual Reality),1.0,645027.0,18.200001,1.359657e+09,103.0,0.009709,0.970874,2475.0,1.700001,5473.0,0.007309
2,2017,Andhra Pradesh,Advertising,3.0,645027.0,18.200001,1.359657e+09,103.0,0.029126,2.912621,2475.0,1.700001,5473.0,0.009867
3,2017,Andhra Pradesh,Agriculture,1.0,645027.0,18.200001,1.359657e+09,103.0,0.009709,0.970874,2475.0,1.700001,5473.0,0.032341
4,2017,Andhra Pradesh,Analytics,1.0,645027.0,18.200001,1.359657e+09,103.0,0.009709,0.970874,2475.0,1.700001,5473.0,0.011511


In [2]:
# State-level yearly startup totals

state_year = (
    df.groupby(["state", "year"], as_index=False)
      .agg(
          startups=("startup_count", "sum"),
          nsva=("nsva", "max")
      )
      .sort_values(["state", "year"])
)

# YoY startup growth
state_year["startup_growth"] = (
    state_year
    .groupby("state")["startups"]
    .pct_change() * 100
)

In [3]:
# Three-year startup CAGR

state_year["startup_cagr_3y"] = (
    state_year
    .groupby("state")["startups"]
    .transform(
        lambda x: (
            (x / x.shift(3)) ** (1/3) - 1
        ) * 100
    )
)

In [4]:
# Industry Share

state_industry = (
    df.groupby(
        ["year", "state", "industry"],
        as_index=False
    )
    .agg(
        industry_startups=("startup_count", "sum")
    )
)

state_totals = (
    state_industry
    .groupby(["year", "state"])["industry_startups"]
    .transform("sum")
)

state_industry["industry_share"] = (
    state_industry["industry_startups"] /
    state_totals
)

In [5]:
# Industry 3-Year Growth

state_industry = state_industry.sort_values(
    ["state", "industry", "year"]
)

state_industry["industry_cagr_3y"] = (
    state_industry
    .groupby(["state", "industry"])["industry_startups"]
    .transform(
        lambda x: (
            (x / x.shift(3)) ** (1/3) - 1
        ) * 100
    )
)

In [6]:
# NSVA GROWTH

state_year["nsva_growth"] = (
    state_year
    .groupby("state")["nsva"]
    .pct_change() * 100
)

In [7]:
# Startup to NSVA Ratio

state_year["startup_nsva_ratio"] = (
    state_year["startups"] /
    state_year["nsva"]
)

In [8]:
# Industry HHI

hhi_data = (
    df.groupby(
        ["state", "industry"],
        as_index=False
    )["startup_count"]
    .sum()
)

state_total = (
    hhi_data
    .groupby("state")["startup_count"]
    .transform("sum")
)

hhi_data["industry_share"] = (
    hhi_data["startup_count"] /
    state_total
)

hhi_data["share_squared"] = (
    hhi_data["industry_share"] ** 2
)

state_hhi = (
    hhi_data
    .groupby("state", as_index=False)
    ["share_squared"]
    .sum()
    .rename(
        columns={
            "share_squared": "industry_hhi"
        }
    )
)

In [9]:
state_year = state_year.merge(
    state_hhi,
    on="state",
    how="left"
)

In [10]:
state_features = state_year[
    [
        "year",
        "state",
        "startups",
        "nsva",
        "startup_growth",
        "startup_cagr_3y",
        "nsva_growth",
        "startup_nsva_ratio",
        "industry_hhi"
    ]
].copy()

state_industry_features = state_industry[
    [
        "year",
        "state",
        "industry",
        "industry_startups",
        "industry_share",
        "industry_cagr_3y"
    ]
].copy()

engine_df = state_industry_features.merge(
    state_features,
    on=["year", "state"],
    how="left"
)

In [11]:
latest_year = engine_df["year"].max()

engine_latest = engine_df[
    engine_df["year"] == latest_year
].copy()

In [12]:
# ============================================================
# UNDERPENETRATION SCORE
# ============================================================

# Economic size percentile at state level
engine_latest["economic_size_percentile"] = (
    engine_latest["nsva"]
    .rank(pct=True) * 100
)

# Industry startup activity percentile
# Calculated across state-industry observations
engine_latest["industry_activity_percentile"] = (
    engine_latest["industry_startups"]
    .rank(pct=True) * 100
)

# Raw underpenetration gap
engine_latest["underpenetration_gap"] = (
    engine_latest["economic_size_percentile"]
    - engine_latest["industry_activity_percentile"]
)

# Convert to 0-100 score
min_gap = engine_latest["underpenetration_gap"].min()
max_gap = engine_latest["underpenetration_gap"].max()

engine_latest["underpenetration_score"] = (
    (
        engine_latest["underpenetration_gap"] - min_gap
    )
    /
    (max_gap - min_gap)
) * 100

In [13]:
# ============================================================
# UNDERPENETRATION SCORE
# ============================================================

# Economic size percentile at state level
engine_df["economic_size_percentile"] = (
    engine_df["nsva"]
    .rank(pct=True) * 100
)

# Industry startup activity percentile
# Calculated across state-industry observations
engine_df["industry_activity_percentile"] = (
    engine_df["industry_startups"]
    .rank(pct=True) * 100
)

# Raw underpenetration gap
engine_df["underpenetration_gap"] = (
    engine_df["economic_size_percentile"]
    - engine_df["industry_activity_percentile"]
)

# Convert to 0-100 score
min_gap = engine_df["underpenetration_gap"].min()
max_gap = engine_df["underpenetration_gap"].max()

engine_df["underpenetration_score"] = (
    (
        engine_df["underpenetration_gap"] - min_gap
    )
    /
    (max_gap - min_gap)
) * 100

In [14]:
selected_industry = "Finance Tech"

industry_view = engine_latest[
    engine_latest["industry"] == selected_industry
].copy()

In [15]:
industry_view["economic_size_percentile"] = (
    industry_view["nsva"]
    .rank(pct=True) * 100
)

industry_view["industry_activity_percentile"] = (
    industry_view["industry_startups"]
    .rank(pct=True) * 100
)

industry_view["underpenetration_gap"] = (
    industry_view["economic_size_percentile"]
    -
    industry_view["industry_activity_percentile"]
)

min_gap = industry_view["underpenetration_gap"].min()
max_gap = industry_view["underpenetration_gap"].max()

industry_view["underpenetration_score"] = (
    (
        industry_view["underpenetration_gap"] - min_gap
    ) /
    (max_gap - min_gap)
) * 100

In [16]:
engine_df.to_csv(
    "data/processed/engine_master_data.csv",
    index=False
)

engine_latest.to_csv(
    "data/processed/engine_latest_data.csv",
    index=False
)


print("Saved successfully:")
print("1. engine_master_data.csv")
print("2. engine_latest_data.csv")

print("\nEngine Master Shape:")
print(engine_df.shape)

print("\nEngine Latest Shape:")
print(engine_latest.shape)

Saved successfully:
1. engine_master_data.csv
2. engine_latest_data.csv

Engine Master Shape:
(8756, 17)

Engine Latest Shape:
(656, 17)
